[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShintaroMinami/notebook_proteina_complexa/blob/main/proteina_complexa.ipynb)

# **Proteina Complexa — Hands-On Notebook**

このノートブックでは、NVIDIAが開発したタンパク質デザインモデル **Proteina Complexa** を
Google Colab上で実行します。

> **必要なもの:** Googleアカウントのみ。ローカル環境の構築は不要です。
>
> **使い方:** 各セル左側の **実行ボタン（▶）** を押して、セルを上から順に実行してください。

In [ ]:
#@title 初期設定
#@markdown このセルを最初に実行してください。
#@markdown
#@markdown ---
#@markdown ### Google Drive の使用
#@markdown - **ON**: 初回はダウンロード後 Drive に保存。2回目以降は Drive から復元（高速）
#@markdown - **OFF**: 毎回インターネットからダウンロード
use_google_drive = False #@param {type:"boolean"}

import subprocess
from IPython.display import display, HTML

def run(cmd, title=""):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    lines = output.split("\n")
    icon = "\u2705" if result.returncode == 0 else "\u274c"
    summary = f"{icon} {title} ({len(lines)} lines)" if title else f"{icon} Output ({len(lines)} lines)"
    html = (
        f'<details><summary style="cursor:pointer;font-weight:bold;padding:4px 0">{summary}</summary>'
        f'<pre style="max-height:300px;overflow-y:auto;background:#f5f5f5;padding:8px;'
        f'font-size:11px;border-radius:4px;margin-top:4px">{output}</pre>'
        f'</details>'
    )
    display(HTML(html))
    return result.returncode

# ---- Complex structure helpers (target + binder PDB; last chain = binder) ----
# Imports are lazy so this cell stays runnable before the install step.
def parse_complex(pdb_path):
    """Parse a target+binder PDB. Returns (structure, target_chain_ids, binder_chain_id)."""
    from Bio.PDB import PDBParser
    structure = PDBParser(QUIET=True).get_structure("complex", str(pdb_path))
    chains = list(structure[0].get_chains())
    return structure, [c.id for c in chains[:-1]], chains[-1].id

def superpose_on_target(des_struct, pred_struct, target_chain_ids):
    """Superimpose pred onto des on matched target CA atoms (modifies pred in place)."""
    from Bio.PDB import Superimposer
    des_ca, pred_ca = [], []
    for des_cid, pred_chain in zip(target_chain_ids, list(pred_struct[0].get_chains())[:-1]):
        des_map = {r.id[1]: r["CA"] for r in des_struct[0][des_cid].get_residues()
                   if r.id[0] == " " and "CA" in r}
        for r in pred_chain.get_residues():
            if r.id[0] == " " and "CA" in r and r.id[1] in des_map:
                des_ca.append(des_map[r.id[1]])
                pred_ca.append(r["CA"])
    if len(des_ca) >= 3:
        sup = Superimposer()
        sup.set_atoms(des_ca, pred_ca)
        sup.apply(pred_struct[0].get_atoms())
    return len(des_ca)

def binder_ca_rmsd(des_struct, pred_struct, des_binder_cid, pred_binder_cid):
    """CA RMSD between design and predicted binder, over the overlapping length."""
    import numpy as np
    des = [r["CA"] for r in des_struct[0][des_binder_cid].get_residues() if r.id[0] == " " and "CA" in r]
    pred = [r["CA"] for r in pred_struct[0][pred_binder_cid].get_residues() if r.id[0] == " " and "CA" in r]
    n = min(len(des), len(pred))
    if n == 0:
        return float("nan")
    d = np.array([a.coord for a in des[:n]]) - np.array([a.coord for a in pred[:n]])
    return float(np.sqrt(np.mean(np.sum(d ** 2, axis=1))))

def binder_sequence(structure, binder_cid):
    """One-letter sequence of the binder chain."""
    from Bio.PDB.Polypeptide import PPBuilder
    peps = PPBuilder().build_peptides(structure[0][binder_cid])
    return "".join(str(pp.get_sequence()) for pp in peps)

def struct_to_pdb(structure):
    """Serialize a Bio.PDB structure to a PDB-format string."""
    import io
    from Bio.PDB import PDBIO
    pdbio = PDBIO()
    pdbio.set_structure(structure)
    stream = io.StringIO()
    pdbio.save(stream)
    return stream.getvalue()

if use_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CACHE = "/content/drive/MyDrive/.proteina_complexa_cache"
    import os
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    print(f"Google Drive cache: {DRIVE_CACHE}")
else:
    DRIVE_CACHE = None
    print("Google Drive: OFF（毎回ダウンロードします）")

print("Ready.")


---
## **1. 環境構築**

はじめに、必要なソフトウェアをインストールします。

In [ ]:
#@title 1.1 GPU の確認
#@markdown GPUが割り当てられているか確認します。割り当てられていない場合は、ランタイムのタイプを変更してください。
#@markdown
#@markdown ---

import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")
else:
    print("GPU が見つかりません。ランタイムのタイプを変更してください。")

print(f"Python: {sys.version}")

In [ ]:
#@title 1.2 ソフトウェアのインストール
#@markdown 必要なソフトウェアを順次インストールします。実行には10〜15分程度かかります。
#@markdown
#@markdown ---

import os

REPO_DIR = "/content/Proteina-Complexa"

# --- uv + repository ---
run("pip install -q uv", "uv install")

if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    run("git clone --depth 1 https://github.com/NVIDIA-Digital-Bio/Proteina-Complexa.git /content/Proteina-Complexa", "git clone")

os.chdir(REPO_DIR)
print(f"リポジトリ: {os.getcwd()}")

# --- PyTorch ---
run(
    "uv pip install --system torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch + CUDA 12.6"
)

# --- Proteina Complexa ---
run("uv pip install --system --index-strategy unsafe-best-match -e .", "Proteina Complexa")
run(
    "uv pip install --system torch_geometric torch_scatter torch_sparse torch_cluster"
    " -f https://data.pyg.org/whl/torch-2.7.0+cu126.html",
    "PyTorch Geometric"
)
run("uv pip install --system graphein==1.7.7 --no-deps", "Graphein")
run("uv pip install --system \"atomworks[ml,openbabel,dev]\" 2>/dev/null || true", "Atomworks")
run("uv pip install --system py3Dmol", "py3Dmol")
run("uv pip install --system biotite==1.6.0", "biotite")

# --- Boltz-2 (before ColabDesign to avoid dependency conflicts) ---
run("uv pip install --system --index-strategy unsafe-best-match \"boltz[cuda]\"", "Boltz-2")
run(
    "uv pip install --system --reinstall torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch reinstall"
)
run("uv pip install --system \"numpy<2.1\"", "NumPy pin")

# --- ColabDesign (AF2) — installed after Boltz to keep correct JAX/Flax versions ---
run("uv pip install --system colabdesign==1.1.1 alphafold-colabfold==2.3.7", "ColabDesign")
import sys
sys.path.insert(0, f"{REPO_DIR}/community_models")
run(
    "uv pip install --system jaxlib==0.4.29+cuda12.cudnn91"
    " -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    "jaxlib (CUDA)"
)
run(
    "uv pip install --system 'jax[cuda12]==0.4.29'"
    " -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    "JAX (CUDA)"
)
run("uv pip install --system flax==0.9.0 --no-deps", "Flax")

import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print("Done.")

In [ ]:
#@title 1.3 モデルファイルをダウンロード
#@markdown ---
#@markdown | モデル | 用途 | サイズ |
#@markdown |---|---|---|
#@markdown | Protein Target | タンパク質に結合するバインダーの設計 | 約 6.6 GB |
#@markdown | Ligand Target | 低分子リガンドに結合するバインダーの設計 | 約 5.9 GB |
#@markdown | AME | モチーフのスキャフォールディング | 約 5.9 GB |
#@markdown | AF2 | 構造予測・検証 | 約 5.4 GB |
#@markdown | Boltz-1 / Boltz-2 | 構造予測・検証 | 約 5 GB |
#@markdown ---
download_protein_target = True #@param {type:"boolean"}
download_ligand_target = False #@param {type:"boolean"}
download_ame = False #@param {type:"boolean"}
download_af2 = True #@param {type:"boolean"}
download_boltz = True #@param {type:"boolean"}

import os, shutil
from pathlib import Path

# --- Proteina Complexa ---
if not os.path.exists(".env"):
    run("complexa init", "Proteina Complexa init")
else:
    print("Proteina Complexa: .env already exists, skipping init.")

os.environ["COMPLEXA_INIT"] = "1"
os.environ["LOCAL_CODE_PATH"] = "/content/Proteina-Complexa"
os.environ["DATA_PATH"] = "/content/Proteina-Complexa/assets"

complexa_models = []
if download_protein_target:
    complexa_models.append("--complexa")
if download_ligand_target:
    complexa_models.append("--complexa-ligand")
if download_ame:
    complexa_models.append("--complexa-ame")

if complexa_models:
    dl_flags = " ".join(complexa_models)
    complexa_cache = Path(DRIVE_CACHE) / "complexa" if DRIVE_CACHE else None
    complexa_local = Path("ckpts")

    if complexa_cache and complexa_cache.exists() and any(complexa_cache.iterdir()):
        # Symlink instead of copying: copying ~6.6 GB over the Drive FUSE mount is very slow
        print("Proteina Complexa weights: linking from Google Drive...")
        if complexa_local.exists() or complexa_local.is_symlink():
            complexa_local.unlink() if complexa_local.is_symlink() else shutil.rmtree(complexa_local)
        complexa_local.symlink_to(complexa_cache)
        print("Proteina Complexa weights: ready (symlink).")
    elif complexa_local.exists() and any(complexa_local.glob("*.ckpt")):
        print("Proteina Complexa weights: already downloaded.")
    else:
        run(f"complexa download {dl_flags}", "Proteina Complexa weights")
        if complexa_cache and complexa_local.exists():
            print("Saving Proteina Complexa weights to Google Drive...")
            if complexa_cache.exists():
                shutil.rmtree(complexa_cache)
            shutil.copytree(complexa_local, complexa_cache)
            shutil.rmtree(complexa_local)
            complexa_local.symlink_to(complexa_cache)
            print("Saved.")

# --- AF2 (AlphaFold2 multimer) ---
AF2_PARAMS_DIR = Path("/content/af2_params")

if download_af2:
    af2_cache = Path(DRIVE_CACHE) / "af2_params" if DRIVE_CACHE else None

    if af2_cache and af2_cache.exists() and any(af2_cache.iterdir()):
        print("AF2 weights: linking from Google Drive...")
        if AF2_PARAMS_DIR.exists() or AF2_PARAMS_DIR.is_symlink():
            AF2_PARAMS_DIR.unlink() if AF2_PARAMS_DIR.is_symlink() else shutil.rmtree(AF2_PARAMS_DIR)
        AF2_PARAMS_DIR.symlink_to(af2_cache)
        print("AF2 weights: ready (symlink).")
    elif AF2_PARAMS_DIR.exists() and any(AF2_PARAMS_DIR.glob("*.npz")):
        print("AF2 weights: already downloaded.")
    else:
        AF2_PARAMS_DIR.mkdir(parents=True, exist_ok=True)
        run("wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar -O /tmp/af2_params.tar", "AF2 weights download")
        run(f"tar -xf /tmp/af2_params.tar -C {AF2_PARAMS_DIR}", "AF2 weights extract")
        run("rm -f /tmp/af2_params.tar")
        if af2_cache:
            print("Saving AF2 weights to Google Drive...")
            if af2_cache.exists():
                shutil.rmtree(af2_cache)
            shutil.copytree(AF2_PARAMS_DIR, af2_cache)
            shutil.rmtree(AF2_PARAMS_DIR)
            AF2_PARAMS_DIR.symlink_to(af2_cache)
            print("Saved.")

# --- Boltz-1 / Boltz-2 ---
if download_boltz:
    boltz_cache = Path(DRIVE_CACHE) / "boltz" if DRIVE_CACHE else None
    boltz_local = Path("/root/.boltz")

    # Link to the Drive cache if present (fast); otherwise prepare a local dir
    if boltz_cache and boltz_cache.exists() and any(boltz_cache.iterdir()):
        print("Boltz weights: linking from Google Drive...")
        if boltz_local.exists() or boltz_local.is_symlink():
            boltz_local.unlink() if boltz_local.is_symlink() else shutil.rmtree(boltz_local)
        boltz_local.symlink_to(boltz_cache)
    else:
        boltz_local.mkdir(parents=True, exist_ok=True)

    # Download only what's missing. A complete cache skips this entirely; it also
    # repairs an older cache that lacks the Boltz-2 CCD components.
    mols = boltz_local / "mols"
    ckpts = ["ccd.pkl", "boltz1_conf.ckpt", "boltz2_conf.ckpt", "boltz2_aff.ckpt"]
    mols_ready = (mols / "ALA.pkl").exists()  # the dir alone may be empty/partial
    if mols_ready and all((boltz_local / f).exists() for f in ckpts):
        print("Boltz weights: ready.")
    else:
        # a partial mols dir makes download_boltz2 skip re-extraction, so clear it first
        if mols.exists() and not mols_ready:
            shutil.rmtree(mols)
        run("python -c \"from pathlib import Path; from boltz.main import download_boltz1, download_boltz2; p = Path('/root/.boltz'); download_boltz1(p); download_boltz2(p)\"", "Boltz weights")

    # If downloaded into a fresh local dir (no cache yet), save it to Drive
    if boltz_cache and not boltz_local.is_symlink():
        print("Saving Boltz weights to Google Drive...")
        if boltz_cache.exists():
            shutil.rmtree(boltz_cache)
        shutil.copytree(boltz_local, boltz_cache)
        shutil.rmtree(boltz_local)
        boltz_local.symlink_to(boltz_cache)
        print("Saved.")

print("All selected models ready.")

In [ ]:
#@title 1.4 インストールの確認
#@markdown ソフトウェアとモデルファイルのインストールが正しく行われたか確認します。
#@markdown
#@markdown ---

from pathlib import Path
import torch

all_ok = True

print("Software")
print("─" * 40)
print(f"  PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

import shutil
if shutil.which("complexa"):
    print("  Proteina Complexa CLI: OK")
else:
    print("  Proteina Complexa CLI: NOT FOUND")
    all_ok = False

try:
    import colabdesign
    ver = getattr(colabdesign, "__version__", "local")
    print(f"  ColabDesign: OK ({ver})")
except ImportError:
    print("  ColabDesign: NOT FOUND")
    all_ok = False

try:
    import jax
    print(f"  JAX: OK ({jax.__version__}, {jax.devices()[0].platform})")
except ImportError:
    print("  JAX: NOT FOUND")
    all_ok = False

print()
print("Model weights")
print("─" * 40)

ckpt_dir = Path("/content/Proteina-Complexa/ckpts")
weight_checks = {
    "Protein Target": ("complexa.ckpt", download_protein_target),
    "Ligand Target": ("complexa_ligand.ckpt", download_ligand_target),
    "AME": ("complexa_ame.ckpt", download_ame),
}

for name, (filename, requested) in weight_checks.items():
    if not requested:
        continue
    path = ckpt_dir / filename
    if path.exists():
        size_gb = path.stat().st_size / (1024**3)
        print(f"  {name}: OK ({size_gb:.1f} GB)")
    else:
        print(f"  {name}: NOT FOUND ({path})")
        all_ok = False

if download_af2:
    if AF2_PARAMS_DIR.exists() and any(AF2_PARAMS_DIR.glob("*multimer_v3*")):
        n_files = len(list(AF2_PARAMS_DIR.glob("*.npz")))
        print(f"  AF2: OK ({n_files} param files)")
    else:
        print(f"  AF2: NOT FOUND ({AF2_PARAMS_DIR})")
        all_ok = False

if download_boltz:
    boltz_dir = Path("/root/.boltz")
    if boltz_dir.exists() and any(boltz_dir.iterdir()):
        total = sum(f.stat().st_size for f in boltz_dir.rglob("*") if f.is_file())
        print(f"  Boltz-1: OK ({total / (1024**3):.1f} GB)")
    else:
        print(f"  Boltz-1: NOT FOUND ({boltz_dir})")
        all_ok = False

print()
if all_ok:
    print("All checks passed!")
else:
    print("Some checks failed. Please re-run the corresponding cells above.")

---
## **2 プロジェクトの準備**

In [ ]:
#@title 2.1 プロジェクトの作成
#@markdown プロジェクト名を入力してください。
#@markdown
#@markdown ---
project_name = "PDL1_binder_design" #@param {type:"string"}

from pathlib import Path

PROJECT_DIR = Path(f"/content/projects/{project_name}")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "targets").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
(PROJECT_DIR / "analysis").mkdir(exist_ok=True)

print(f"Project: {project_name}")
print(f"Directory: {PROJECT_DIR}")
print()
print(f"  targets/   ... ターゲットPDBファイル")
print(f"  results/   ... デザイン結果")
print(f"  analysis/  ... 解析結果")

In [ ]:
#@title 2.2 ターゲットの指定
#@markdown - **PDB ID**: RCSB PDBからダウンロードします（例: `3BIK`, `7BQD`）
#@markdown - **Upload**: 手元のPDBファイルをアップロードします
#@markdown
#@markdown ---
target_source = "PDB ID" #@param ["PDB ID", "Upload"]
pdb_id = "3BIK" #@param {type:"string"}

from pathlib import Path
from Bio.PDB import PDBParser
import py3Dmol
import warnings
warnings.filterwarnings("ignore")

target_dir = PROJECT_DIR / "targets"

if target_source == "PDB ID":
    pdb_id = pdb_id.strip().upper()
    target_pdb = target_dir / f"{pdb_id}.pdb"
    if target_pdb.exists():
        print(f"Already downloaded: {target_pdb}")
    else:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        run(f"wget -q -O {target_pdb} {url}", f"Download {pdb_id}")
        if target_pdb.exists() and target_pdb.stat().st_size > 0:
            print(f"Downloaded: {target_pdb}")
        else:
            print(f"Error: PDB ID '{pdb_id}' が見つかりませんでした。IDを確認してください。")
else:
    from google.colab import files
    print("PDBファイルを選択してください:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        target_pdb = target_dir / filename
        target_pdb.write_bytes(content)
        pdb_id = target_pdb.stem.upper()
        print(f"Uploaded: {target_pdb}")

# Chain info
parser = PDBParser(QUIET=True)
structure = parser.get_structure("target", str(target_pdb))

print(f"\nFile: {target_pdb.name}")
print("─" * 50)
for model in structure:
    for chain in model:
        residues = [r for r in chain.get_residues() if r.id[0] == " "]
        if not residues:
            continue
        first = residues[0].id[1]
        last = residues[-1].id[1]
        print(f"  Chain {chain.id}: residues {first}-{last} ({len(residues)} residues)")
    break
print("─" * 50)

# 3D viewer — color by chain
CHAIN_COLORS = [
    "#3498db", "#2ecc71", "#e67e22", "#9b59b6", "#e74c3c",
    "#1abc9c", "#f1c40f", "#34495e", "#e91e63", "#00bcd4",
]

with open(target_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")

chain_ids = [ch.id for ch in structure[0] if any(r.id[0] == " " for r in ch)]
for i, cid in enumerate(chain_ids):
    color = CHAIN_COLORS[i % len(CHAIN_COLORS)]
    view.setStyle({"chain": cid}, {"cartoon": {"color": color}})

view.zoomTo()
view.show()

print("Chains: " + ", ".join(f"{cid}={CHAIN_COLORS[i % len(CHAIN_COLORS)]}" for i, cid in enumerate(chain_ids)))

In [ ]:
#@title 2.3 設計パラメータの設定
#@markdown 設計に使用するターゲットのチェーンと残基範囲、ホットスポット残基、生成するバインダーの長さを指定します。
#@markdown
#@markdown ---
#@markdown ### ターゲット設定
#@markdown - **target_chain_residues**: ターゲットのチェーンと残基範囲（例: `A1-115`）
#@markdown - **hotspot_residues**: バインダーが結合してほしい残基（カンマ区切り、例: `A56,A113,A121`）
#@markdown ### バインダー設定
#@markdown - **binder_length_min / max**: 生成するバインダーの長さの範囲
#@markdown ---
target_chain_residues = "A18-133" #@param {type:"string"}
hotspot_residues = "A56,A113,A115,A121,A123" #@param {type:"string"}
binder_length_min = 60 #@param {type:"integer"}
binder_length_max = 80 #@param {type:"integer"}

import re
from pathlib import Path

# --- Parse user input ---
match = re.match(r"([A-Z])(\d+)-(\d+)", target_chain_residues)
if not match:
    raise ValueError(f"Invalid format: '{target_chain_residues}'. Expected: A18-133")
target_chain = match.group(1)
target_resi_start = int(match.group(2))
target_resi_end = int(match.group(3))

hotspot_input = [r.strip() for r in hotspot_residues.split(",") if r.strip()]
hotspot_resis_orig = []
for h in hotspot_input:
    m = re.match(r"([A-Z])(\d+)", h)
    if m:
        hotspot_resis_orig.append(int(m.group(2)))

# --- Crop target region from original PDB and renumber from 1 ---
original_target_pdb = target_pdb

with open(original_target_pdb) as f:
    pdb_lines = f.readlines()

seen_residues = []
for line in pdb_lines:
    if not line.startswith("ATOM"):
        continue
    if line[21] != target_chain:
        continue
    resseq = int(line[22:26])
    icode = line[26]
    if resseq < target_resi_start or resseq > target_resi_end:
        continue
    key = (resseq, icode)
    if key not in seen_residues:
        seen_residues.append(key)

if not seen_residues:
    raise ValueError(
        f"No residues found for chain {target_chain} "
        f"in range {target_resi_start}-{target_resi_end}"
    )

residue_map = {}
for new_idx, key in enumerate(seen_residues, start=1):
    residue_map[key] = new_idx
n_residues = len(seen_residues)

hotspot_resis = []
for orig_resi in hotspot_resis_orig:
    new_resi = None
    for (rseq, _icode), nrseq in residue_map.items():
        if rseq == orig_resi:
            new_resi = nrseq
            break
    if new_resi is not None:
        hotspot_resis.append(new_resi)
    else:
        print(f"  Warning: hotspot residue {target_chain}{orig_resi} not found in target region")

processed_pdb = target_dir / f"{pdb_id}_target.pdb"
serial = 1
with open(processed_pdb, "w") as f:
    for line in pdb_lines:
        if not line.startswith("ATOM"):
            continue
        if line[21] != target_chain:
            continue
        resseq = int(line[22:26])
        icode = line[26]
        key = (resseq, icode)
        if key not in residue_map:
            continue
        new_resseq = residue_map[key]
        f.write(f"ATOM  {serial:5d}{line[11:21]}{target_chain}{new_resseq:4d} {line[27:]}")
        serial += 1
    f.write(f"TER   {serial:5d}\n")
    f.write("END\n")

target_pdb = processed_pdb

# --- Build target config for Complexa ---
hotspot_list = [f"{target_chain}{r}" for r in hotspot_resis]

target_config = {
    "source": "user_targets",
    "target_filename": processed_pdb.stem,
    "target_path": str(processed_pdb),
    "target_input": f"{target_chain}1-{n_residues}",
    "hotspot_residues": hotspot_list,
    "binder_length": [binder_length_min, binder_length_max],
    "pdb_id": pdb_id.lower(),
}

TASK_NAME = f"user_{pdb_id}"

print("Target configuration:")
for k, v in target_config.items():
    print(f"  {k}: {v}")
print(f"  task_name: {TASK_NAME}")

print(f"\nProcessed target PDB: {processed_pdb.name}")
print(f"  {n_residues} residues (chain {target_chain}, renumbered 1-{n_residues})")
print(f"  Original: {target_chain_residues}  →  {target_chain}1-{n_residues}")
if hotspot_resis:
    mapping_str = ", ".join(
        f"{target_chain}{orig}→{target_chain}{new}"
        for orig, new in zip(hotspot_resis_orig, hotspot_resis)
    )
    print(f"  Hotspot:  {mapping_str}")

# --- 3D Viewer (original PDB with target region highlighted) ---
import py3Dmol

with open(original_target_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")

view.setStyle(
    {"cartoon": {"color": "lightgray", "opacity": 0.5}}
)
view.setStyle(
    {"chain": target_chain, "resi": [f"{target_resi_start}-{target_resi_end}"]},
    {"cartoon": {"color": "skyblue"}}
)
if hotspot_resis_orig:
    view.setStyle(
        {"chain": target_chain, "resi": hotspot_resis_orig},
        {"cartoon": {"color": "red"}, "stick": {"color": "red"}}
    )

view.zoomTo({"chain": target_chain, "resi": [f"{target_resi_start}-{target_resi_end}"]})
view.show()

print(f"\nBlue: target region ({target_chain_residues})")
print(f"Red:  hotspot residues ({hotspot_residues})")
print(f"Gray: non-target region")

---
## **3. バインダーデザインの実行**

In [ ]:
#@title 3.1 デザインパイプラインの設定
#@markdown - **num_samples**: 最終的に得られる生成デザインの合計数
#@markdown - **batch_size**: 一度の処理で用いるバッチサイズ
#@markdown - **search_algorithm**: 探索アルゴリズム（デモでは single-pass 推奨）
#@markdown - **seed**: 乱数シード（再現性のため）
#@markdown - **ncpus**: 使用するCPUコア数
#@markdown
#@markdown ---
num_samples = 8 #@param {type:"integer"}
batch_size = 1 #@param {type:"integer"}
search_algorithm = "single-pass" #@param ["single-pass", "best-of-n"]
seed = 1101 #@param {type:"integer"}
ncpus = 2 #@param {type:"integer"}

from pathlib import Path

COMPLEXA_DIR = Path("/content/Proteina-Complexa")
SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
LOGS_DIR = PROJECT_DIR / "logs"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

pipeline_yaml = f"""defaults:
  - pipeline/binder/binder_generate@generation
  - _self_

run_name: {project_name}
ckpt_path: {COMPLEXA_DIR / "ckpts"}
ckpt_name: complexa.ckpt
autoencoder_ckpt_path: {COMPLEXA_DIR / "ckpts" / "complexa_ae.ckpt"}
ncpus_: {ncpus}
seed: {seed}
gen_njobs: 1
eval_njobs: 1

hydra:
  run:
    dir: {LOGS_DIR}
"""

pipeline_path = COMPLEXA_DIR / "configs" / "user_pipeline.yaml"
pipeline_path.write_text(pipeline_yaml)
print(f"Pipeline config written: {pipeline_path}")
print(f"Samples: {num_samples}, Batch size: {batch_size}, Algorithm: {search_algorithm}, Seed: {seed}")
print(f"Output: {PROJECT_DIR}")


In [ ]:
#@title 3.2 デザインの実行
#@markdown T4 GPUでは数分から十数分かかります。
#@markdown
#@markdown ---
import os

os.chdir("/content/Proteina-Complexa")

# Clear previous results to allow re-runs
import shutil
if SAMPLES_DIR.exists():
    shutil.rmtree(SAMPLES_DIR)
    SAMPLES_DIR.mkdir(parents=True)

cmd = (
    f"complexa generate configs/user_pipeline.yaml"
    f" ++root_path={SAMPLES_DIR}"
    f" ++generation.task_name={TASK_NAME}"
    f" ++generation.search.algorithm={search_algorithm}"
    f" ++generation.dataloader.batch_size={batch_size}"
    f" ++generation.dataloader.dataset.nres.nsamples={num_samples}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.source=user_targets"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_filename={target_config['target_filename']}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_path={target_config['target_path']}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_input={target_config['target_input']}"
    f" '++generation.target_dict_cfg.{TASK_NAME}.hotspot_residues={target_config["hotspot_residues"]}'"
    f" '++generation.target_dict_cfg.{TASK_NAME}.binder_length={target_config["binder_length"]}'"
    f" ++generation.target_dict_cfg.{TASK_NAME}.pdb_id={target_config['pdb_id']}"
    f" ++generation.reward_model=null"
)

print(f"Running: {cmd[:100]}...")
print()
run(cmd, "Binder generation")

# --- Results ---
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb")) if SAMPLES_DIR.exists() else []

if pdb_files:
    print(f"\nGenerated {len(pdb_files)} binder candidates:")
    for p in pdb_files:
        print(f"  {p.relative_to(SAMPLES_DIR)}")
    print(f"\nOutput: {SAMPLES_DIR}")
else:
    print("\nPDB files not found. Check the generation log above for errors.")
    print(f"Searched: {SAMPLES_DIR}")

# Free GPU memory after generation
import gc
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
#@title 3.3 デザイン結果の3D表示
#@markdown 生成したバインダー候補を一覧表示します。
#@markdown
#@markdown ---
#@markdown - **max_cols**: グリッドの最大列数（1–8）
#@markdown - **linked_view**: ON にすると全ビューアが連動して回転・ズームします
#@markdown ---
max_cols = 4 #@param {type:"integer"}
linked_view = True #@param {type:"boolean"}

import math
import py3Dmol
from pathlib import Path

SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb"))

if not pdb_files:
    print("PDBファイルが見つかりません。先に 3.2 を実行してください。")
else:
    n = len(pdb_files)
    cols = min(max_cols, n, 8)
    rows = min(math.ceil(n / cols), 8)
    n_show = rows * cols

    view = py3Dmol.view(viewergrid=(rows, cols), width=250 * cols, height=280 * rows, linked=linked_view)

    for idx, pdb_path in enumerate(pdb_files[:n_show]):
        r, c = divmod(idx, cols)
        with open(pdb_path) as f:
            pdb_data = f.read()
        view.addModel(pdb_data, "pdb", viewer=(r, c))

        _, target_cids, binder_cid = parse_complex(pdb_path)

        # Target: white
        for cid in target_cids:
            view.setStyle({"chain": cid}, {"cartoon": {"color": "white"}}, viewer=(r, c))
        # Hotspot: red cartoon + sticks
        if hotspot_resis:
            view.setStyle(
                {"chain": target_chain, "resi": hotspot_resis},
                {"cartoon": {"color": "red"}, "stick": {"color": "red"}},
                viewer=(r, c),
            )
        # Binder: green
        view.setStyle({"chain": binder_cid}, {"cartoon": {"color": "lightgreen"}}, viewer=(r, c))

        view.addLabel(pdb_path.stem, {
            "position": {"x": 0, "y": 0, "z": 0},
            "backgroundColor": "black", "fontColor": "white",
            "fontSize": 10, "backgroundOpacity": 0.6,
            "inFront": True,
        }, viewer=(r, c))
        view.zoomTo(viewer=(r, c))

    view.show()

    mode = "linked" if linked_view else "independent"
    print(f"Showing {min(n, n_show)}/{n} designs  (White: target, Red: hotspot, Green: binder, {mode})")
    if n > n_show:
        print(f"  残り {n - n_show} 件は表示されていません。max_cols を増やすか、グリッドを分割してください。")

---
## **4. 構造バリデーション**

設計したバインダーの配列を構造予測モデル（AF2-Multimer / Boltz-1 / Boltz-2）でリフォールディングし、
設計構造との自己一貫性を検証します。

In [ ]:
#@title 4.1 構造バリデーション
#@markdown 設計したバインダーをリフォールディングし、自己一貫性を検証します。
#@markdown
#@markdown ---
#@markdown **注意**: Boltz-1 / Boltz-2 を使用する場合、MSA計算のためにターゲットチェインのアミノ酸配列が
#@markdown 外部サーバー（api.colabfold.com）に送信されます。
#@markdown
#@markdown ---
use_af2 = True #@param {type:"boolean"}
use_boltz1 = False #@param {type:"boolean"}
use_boltz2 = False #@param {type:"boolean"}

import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from Bio.PDB.Polypeptide import PPBuilder
import warnings
warnings.filterwarnings("ignore")

# Proteina-Complexa uses a src layout; its editable install is not always on the
# Colab kernel's sys.path, so add it explicitly before importing.
_REPO_SRC = "/content/Proteina-Complexa/src"
if os.path.isdir(_REPO_SRC) and _REPO_SRC not in sys.path:
    sys.path.insert(0, _REPO_SRC)

try:
    from proteinfoundation.metrics.ipsae import complex_ipSAE
    _HAS_IPSAE = True
    _IPSAE_IMPORT_ERR = ""
except Exception as e:
    _HAS_IPSAE = False
    _IPSAE_IMPORT_ERR = f"{type(e).__name__}: {e}"

os.chdir("/content/Proteina-Complexa")

SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb"))

if not use_af2 and not use_boltz1 and not use_boltz2:
    print("AF2 / Boltz-1 / Boltz-2 のいずれかを選択してください。")
elif not pdb_files:
    print("PDB ファイルが見つかりません。先に 3.2 を実行してください。")
else:
    ppb = PPBuilder()

    # Collect diagnostic messages (warnings/errors) to a downloadable log file
    log_lines = []
    def log_line(msg):
        print(msg)
        log_lines.append(str(msg))

    def compute_tab_rmsd_and_plddt(des_struct, pred_pdb_path, target_chain_ids, binder_chain_id):
        pred_struct, _, pred_bcid = parse_complex(pred_pdb_path)
        bfs = [r["CA"].get_bfactor() for r in pred_struct[0][pred_bcid].get_residues()
               if r.id[0] == " " and "CA" in r]
        plddt_b = float(np.mean(bfs)) / 100.0 if bfs else 0.0
        superpose_on_target(des_struct, pred_struct, target_chain_ids)
        tab_rmsd = binder_ca_rmsd(des_struct, pred_struct, binder_chain_id, pred_bcid)
        return tab_rmsd, plddt_b

    def compute_ipsae(pae_matrix, pdb_path, pae_cutoff):
        """Compute ipSAE using proteinfoundation's complex_ipSAE."""
        if not _HAS_IPSAE:
            return float("nan")
        try:
            # astype(float32) handles bfloat16 PAE from ColabDesign (use_bfloat16=True)
            pae_np = np.asarray(pae_matrix).astype(np.float32)
            if np.isnan(pae_np).any():
                log_line(f"  Warning: PAE matrix contains NaN values")
            pae_tensor = torch.tensor(pae_np, dtype=torch.float32)
            result = complex_ipSAE(
                pae_tensor, str(pdb_path),
                interaction_cutoff=8.0, pae_cutoff=pae_cutoff,
            )
            return result["min"]
        except Exception as e:
            log_line(f"  Warning: ipSAE computation failed: {type(e).__name__}: {e}")
            return float("nan")

    # Parse all design structures
    samples = []
    for pdb_path in pdb_files:
        des_struct, target_cids, binder_cid = parse_complex(pdb_path)
        binder_seq = binder_sequence(des_struct, binder_cid)
        samples.append({
            "pdb_path": pdb_path, "name": pdb_path.stem,
            "des_struct": des_struct, "binder_cid": binder_cid,
            "target_cids": target_cids, "binder_seq": binder_seq,
            "length": len(binder_seq),
        })

    results = {s["name"]: {"name": s["name"], "length": s["length"], "binder_sequence": s["binder_seq"]} for s in samples}

    import contextlib
    @contextlib.contextmanager
    def suppress_output():
        """Silence verbose ColabDesign / library chatter on stdout & stderr."""
        with open(os.devnull, "w") as devnull, \
                contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield

    if not _HAS_IPSAE:
        log_line(f"Note: complex_ipSAE が利用できません（{_IPSAE_IMPORT_ERR}）。"
                 f"AF2 は組み込み ipSAE で代替、Boltz は ipSAE=NaN になります。")

    # ── AF2-Multimer ──
    if use_af2:
        from colabdesign import mk_afdesign_model, clear_mem

        AF2_OUTPUT = PROJECT_DIR / "results" / "af2"
        AF2_PAE_DIR = AF2_OUTPUT / "pae"
        AF2_OUTPUT.mkdir(parents=True, exist_ok=True)
        AF2_PAE_DIR.mkdir(parents=True, exist_ok=True)

        print("AF2-Multimer validation")

        for i, s in enumerate(samples):
            name = s["name"]
            print(f"  [{i+1}/{len(samples)}] {name:<34} ", end="", flush=True)
            try:
                with suppress_output():
                    clear_mem()
                    af_model = mk_afdesign_model(
                        protocol="binder", use_multimer=True, num_recycles=3,
                        data_dir=str(AF2_PARAMS_DIR),
                    )
                    af_model.prep_inputs(
                        pdb_filename=str(s["pdb_path"]),
                        chain=",".join(s["target_cids"]),
                        binder_len=s["length"],
                    )
                    af_model.predict(seq=s["binder_seq"], models=[0], num_recycles=3)

                pred_pdb = AF2_OUTPUT / f"{name}_af2.pdb"
                af_model.save_pdb(str(pred_pdb))

                log = af_model.aux["log"]
                plddt_b = float(log.get("plddt", 0))
                iptm = float(log.get("i_ptm", 0))
                ipae = float(log.get("i_pae", 0)) * 31.0  # ColabDesign logs i_pae as PAE/31; restore Å
                tab_rmsd, _ = compute_tab_rmsd_and_plddt(
                    s["des_struct"], pred_pdb, s["target_cids"], s["binder_cid"]
                )

                # Save PAE matrix (float32) for download / offline verification
                pae_arr = np.asarray(af_model.aux["pae"]).astype(np.float32)
                np.save(AF2_PAE_DIR / f"{name}_af2_pae.npy", pae_arr)

                # ipSAE via complex_ipSAE (pae_cutoff=15.0); fall back to built-in if it fails
                ipsae = compute_ipsae(pae_arr, pred_pdb, pae_cutoff=15.0)
                ipsae_log = float(log.get("min_ipsae", float("nan")))  # ColabDesign built-in (cross-check)
                if np.isnan(ipsae) and not np.isnan(ipsae_log):
                    ipsae = ipsae_log
                    if _HAS_IPSAE:  # a genuine per-sample failure (not just "unavailable")
                        log_lines.append(f"  {name}: complex_ipSAE failed; used built-in ipSAE")

                results[name].update({
                    "af2_pLDDT_binder": plddt_b, "af2_ipTM": iptm,
                    "af2_ipAE": ipae, "af2_ipSAE": ipsae,
                    "af2_ipSAE_check": ipsae_log, "af2_TaB_RMSD": tab_rmsd,
                })
                print(f"ipTM={iptm:.3f} ipSAE={ipsae:.3f}(log {ipsae_log:.3f}) ipAE={ipae:.1f}Å TaB_RMSD={tab_rmsd:.2f}Å")

            except Exception as e:
                print(f"ERROR: {e}")
                log_lines.append(f"  [{i+1}/{len(samples)}] {name}  ERROR: {e}")
                results[name].update({
                    "af2_pLDDT_binder": float("nan"), "af2_ipTM": float("nan"),
                    "af2_ipAE": float("nan"), "af2_ipSAE": float("nan"),
                    "af2_ipSAE_check": float("nan"), "af2_TaB_RMSD": float("nan"),
                })

    # Free AF2 (JAX) GPU memory before Boltz (PyTorch)
    if use_af2:
        try:
            clear_mem()
        except Exception:
            pass
        try:
            import jax
            backend = jax.lib.xla_bridge.get_backend()
            for buf in backend.live_buffers():
                buf.delete()
        except Exception:
            pass
        import gc
        gc.collect()

    # ── Boltz (boltz1 / boltz2 share the same pipeline) ──
    def run_boltz(model_name, prefix, label):
        BOLTZ_DIR = PROJECT_DIR / "results" / prefix
        BOLTZ_INPUT = BOLTZ_DIR / "input"
        BOLTZ_PAE_DIR = BOLTZ_DIR / "pae"
        BOLTZ_INPUT.mkdir(parents=True, exist_ok=True)
        BOLTZ_PAE_DIR.mkdir(parents=True, exist_ok=True)

        print(f"\n{label} validation")
        print("  注意: ターゲット配列を外部サーバー（api.colabfold.com）に送信します。")

        # Prepare YAML: target chains get MSA (server), binder gets msa: empty
        for s in samples:
            yaml_lines = ["version: 1", "sequences:"]
            for chain in s["des_struct"][0]:
                peps = ppb.build_peptides(chain)
                if not peps:
                    continue
                seq = "".join(str(pp.get_sequence()) for pp in peps)
                yaml_lines.append("  - protein:")
                yaml_lines.append(f"      id: {chain.id}")
                yaml_lines.append(f"      sequence: {seq}")
                if chain.id == s["binder_cid"]:
                    yaml_lines.append("      msa: empty")
            (BOLTZ_INPUT / f"{s['name']}.yaml").write_text("\n".join(yaml_lines) + "\n")

        print(f"Validating {len(samples)} samples...")
        # Boltz custom CUDA kernels need Ampere+ (sm_80); disable on older GPUs (e.g. T4) where they hang
        major = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
        kernels_flag = "" if major >= 8 else " --no_kernels"
        cmd = (
            f"boltz predict {BOLTZ_INPUT}"
            f" --model {model_name}"
            f" --use_msa_server"
            f" --out_dir {BOLTZ_DIR}"
            f" --recycling_steps 3"
            f" --diffusion_samples 1"
            f" --output_format pdb"
            f" --write_full_pae"
            f" --num_workers 0"
            f"{kernels_flag}"
            f" --override"
        )
        rc = run(cmd, f"{label} structure prediction")

        pred_dir = BOLTZ_DIR / f"boltz_results_{BOLTZ_INPUT.stem}" / "predictions"
        if rc != 0 or not pred_dir.exists() or not any(pred_dir.iterdir()):
            log_line(f"  Warning: {label} の予測出力がありません (exit={rc})。上の折りたたみログを確認してください。")
        for s in samples:
            name = s["name"]
            sample_dir = pred_dir / name
            conf_files = sorted(sample_dir.glob("confidence_*.json")) if sample_dir.exists() else []
            pred_pdbs = sorted(sample_dir.glob("*.pdb")) if sample_dir.exists() else []
            pae_npzs = sorted(sample_dir.glob("pae_*.npz")) if sample_dir.exists() else []

            if conf_files and pred_pdbs:
                with open(conf_files[0]) as f:
                    conf = json.load(f)
                iptm = float(conf.get("iptm", 0))
                tab_rmsd, plddt_b = compute_tab_rmsd_and_plddt(
                    s["des_struct"], pred_pdbs[0], s["target_cids"], s["binder_cid"]
                )
                scores = {
                    f"{prefix}_pLDDT_binder": plddt_b, f"{prefix}_ipTM": iptm,
                    f"{prefix}_TaB_RMSD": tab_rmsd,
                }
                # ipSAE via complex_ipSAE (PAE from pae_*.npz, pae_cutoff=10.0)
                if pae_npzs:
                    pae_arr = np.load(pae_npzs[0])["pae"].astype(np.float32)
                    np.save(BOLTZ_PAE_DIR / f"{name}_{prefix}_pae.npy", pae_arr)
                    scores[f"{prefix}_ipSAE"] = compute_ipsae(pae_arr, pred_pdbs[0], pae_cutoff=10.0)
                else:
                    log_line(f"  Warning: PAE npz not found for {name} ({label})")
                results[name].update(scores)
            else:
                results[name].update({
                    f"{prefix}_pLDDT_binder": float("nan"), f"{prefix}_ipTM": float("nan"),
                    f"{prefix}_TaB_RMSD": float("nan"),
                })
                if _HAS_IPSAE:
                    results[name][f"{prefix}_ipSAE"] = float("nan")

        # Free Boltz (PyTorch) GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        import gc
        gc.collect()

    if use_boltz1:
        run_boltz("boltz1", "boltz1", "Boltz-1")
    if use_boltz2:
        run_boltz("boltz2", "boltz2", "Boltz-2")

    # ── Save CSV ──
    df = pd.DataFrame(list(results.values()))
    csv_path = PROJECT_DIR / "results" / "validation_scores.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nCSV saved: {csv_path}")

    # Save diagnostic log (included in the results download)
    log_path = PROJECT_DIR / "results" / "validation_log.txt"
    log_path.write_text("\n".join(log_lines) + "\n" if log_lines else "No warnings or errors.\n")
    print(f"Log saved: {log_path}  ({len(log_lines)} message(s))")

    # ── Summary tables ──
    if use_af2:
        print("\n  [AF2-Multimer]  (ipSAE_chk = ColabDesign built-in, for cross-check)")
        print(f"  {'Sample':<25} {'len':>4} {'pLDDT_b':>7} {'ipTM':>6} {'ipAE':>6} {'ipSAE':>6} {'ipSAE_chk':>9} {'TaB_RMSD':>8}")
        print("  " + "─" * 80)
        for r in results.values():
            rmsd = r.get("af2_TaB_RMSD", float("nan"))
            rmsd_s = f"{rmsd:.2f} Å" if not np.isnan(rmsd) else "  N/A"
            print(f"  {r['name']:<25} {r['length']:>4}"
                  f" {r.get('af2_pLDDT_binder', float('nan')):>7.2f}"
                  f" {r.get('af2_ipTM', float('nan')):>6.3f}"
                  f" {r.get('af2_ipAE', float('nan')):>6.2f}"
                  f" {r.get('af2_ipSAE', float('nan')):>6.3f}"
                  f" {r.get('af2_ipSAE_check', float('nan')):>9.3f}"
                  f" {rmsd_s:>8}")

    def print_boltz_summary(prefix, label):
        has_ipsae_col = any(f"{prefix}_ipSAE" in r for r in results.values())
        print(f"\n  [{label}]")
        if has_ipsae_col:
            print(f"  {'Sample':<25} {'len':>4} {'pLDDT_b':>7} {'ipTM':>6} {'ipSAE':>6} {'TaB_RMSD':>8}")
            print("  " + "─" * 64)
        else:
            print(f"  {'Sample':<25} {'len':>4} {'pLDDT_b':>7} {'ipTM':>6} {'TaB_RMSD':>8}")
            print("  " + "─" * 56)
        for r in results.values():
            rmsd = r.get(f"{prefix}_TaB_RMSD", float("nan"))
            rmsd_s = f"{rmsd:.2f} Å" if not np.isnan(rmsd) else "  N/A"
            line = (f"  {r['name']:<25} {r['length']:>4}"
                    f" {r.get(f'{prefix}_pLDDT_binder', float('nan')):>7.2f}"
                    f" {r.get(f'{prefix}_ipTM', float('nan')):>6.3f}")
            if has_ipsae_col:
                line += f" {r.get(f'{prefix}_ipSAE', float('nan')):>6.3f}"
            line += f" {rmsd_s:>8}"
            print(line)

    if use_boltz1:
        print_boltz_summary("boltz1", "Boltz-1")
    if use_boltz2:
        print_boltz_summary("boltz2", "Boltz-2")

    print(f"\n  閾値の調整は 4.3 フィルタリング で行います。")


In [ ]:
#@title 4.2 スコアの解析
#@markdown バリデーション結果を可視化します。
#@markdown ---

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

csv_path = PROJECT_DIR / "results" / "validation_scores.csv"

if not csv_path.exists():
    print("validation_scores.csv が見つかりません。先に 4.1 を実行してください。")
else:
    df = pd.read_csv(csv_path)
    has_af2 = "af2_ipTM" in df.columns
    has_boltz1 = "boltz1_ipTM" in df.columns

    methods = []
    if has_af2:
        methods.append(("AF2", "af2"))
    if has_boltz1:
        methods.append(("Boltz-1", "boltz1"))

    # ── 1. Score table ──
    show_cols = ["name", "length"]
    for _, pfx in methods:
        if pfx == "af2":
            show_cols += [f"{pfx}_pLDDT_binder", f"{pfx}_ipTM", f"{pfx}_ipAE", f"{pfx}_ipSAE", f"{pfx}_TaB_RMSD"]
        else:
            cols = [f"{pfx}_pLDDT_binder", f"{pfx}_ipTM"]
            if f"{pfx}_ipSAE" in df.columns:
                cols.append(f"{pfx}_ipSAE")
            cols.append(f"{pfx}_TaB_RMSD")
            show_cols += cols
    show_cols = [c for c in show_cols if c in df.columns]
    display(df[show_cols].style.format(precision=3, na_rep="N/A"))

    # ── 2. Scatter: ipSAE vs TaB_RMSD ──
    n_m = len(methods)
    fig, axes = plt.subplots(1, n_m, figsize=(5.5 * n_m, 4.5), squeeze=False)

    for idx, (label, pfx) in enumerate(methods):
        ax = axes[0, idx]
        y = df[f"{pfx}_TaB_RMSD"]

        if f"{pfx}_ipSAE" in df.columns:
            x = df[f"{pfx}_ipSAE"]
            x_label = "ipSAE"
        else:
            x = df[f"{pfx}_ipTM"]
            x_label = "ipTM"

        ax.scatter(x, y, s=70, c="steelblue", edgecolors="black", linewidth=0.5, zorder=3)

        for i, nm in enumerate(df["name"]):
            short = nm[:18] + "..." if len(nm) > 18 else nm
            ax.annotate(short, (x.iloc[i], y.iloc[i]),
                        fontsize=7, textcoords="offset points", xytext=(5, 5))

        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel("TaB_RMSD (Å)", fontsize=11)
        ax.set_title(f"{label}: {x_label} vs TaB_RMSD", fontsize=12, fontweight="bold")
        if y.notna().any():
            ax.set_ylim(bottom=-0.2)

    plt.tight_layout()
    plt.show()

    # ── 3. Per-sample metric bars ──
    for label, pfx in methods:
        metric_cols = [("pLDDT_binder", "#3498db"), ("ipTM", "#2ecc71")]

        fig, ax = plt.subplots(figsize=(max(6, len(df) * 1.2), 3.5))
        x_pos = np.arange(len(df))
        width = 0.35

        for j, (m, color) in enumerate(metric_cols):
            col = f"{pfx}_{m}"
            if col in df.columns:
                offset = (j - len(metric_cols) / 2 + 0.5) * width
                ax.bar(x_pos + offset, df[col], width, label=m, color=color, alpha=0.85, edgecolor="white")

        ax.set_xticks(x_pos)
        ax.set_xticklabels([n[:20] for n in df["name"]], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Score", fontsize=11)
        ax.set_title(f"{label}: Per-sample Metrics", fontsize=12, fontweight="bold")
        ax.legend(loc="upper right", fontsize=9)
        ax.set_ylim(0, 1.08)
        plt.tight_layout()
        plt.show()

    # ── 4. AF2 vs Boltz-1 comparison ──
    if has_af2 and has_boltz1:
        fig, axes = plt.subplots(1, 2, figsize=(9, 4))

        ax = axes[0]
        ax.scatter(df["af2_ipTM"], df["boltz1_ipTM"], s=60, c="steelblue", edgecolors="black", lw=0.5)
        ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
        for i, nm in enumerate(df["name"]):
            ax.annotate(nm[:15], (df["af2_ipTM"].iloc[i], df["boltz1_ipTM"].iloc[i]),
                        fontsize=7, textcoords="offset points", xytext=(4, 4))
        ax.set_xlabel("AF2 ipTM", fontsize=11)
        ax.set_ylabel("Boltz-1 ipTM", fontsize=11)
        ax.set_title("ipTM: AF2 vs Boltz-1", fontsize=12, fontweight="bold")
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)

        ax = axes[1]
        mask = df["af2_TaB_RMSD"].notna() & df["boltz1_TaB_RMSD"].notna()
        if mask.any():
            ax.scatter(df.loc[mask, "af2_TaB_RMSD"], df.loc[mask, "boltz1_TaB_RMSD"],
                       s=60, c="coral", edgecolors="black", lw=0.5)
            lim = max(df.loc[mask, "af2_TaB_RMSD"].max(), df.loc[mask, "boltz1_TaB_RMSD"].max()) * 1.2
            ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.4)
            for i in df[mask].index:
                ax.annotate(df["name"].iloc[i][:15],
                            (df["af2_TaB_RMSD"].iloc[i], df["boltz1_TaB_RMSD"].iloc[i]),
                            fontsize=7, textcoords="offset points", xytext=(4, 4))
        ax.set_xlabel("AF2 TaB_RMSD (Å)", fontsize=11)
        ax.set_ylabel("Boltz-1 TaB_RMSD (Å)", fontsize=11)
        ax.set_title("TaB_RMSD: AF2 vs Boltz-1", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
#@title 4.3 フィルタリング
#@markdown 4.2 の結果を参考に閾値を調整し、候補をフィルタリングします。
#@markdown
#@markdown ---
#@markdown - **ipSAE** (interaction prediction Score from Aligned Errors): 結合界面の予測構造信頼度（高いほど良い、0–1）
#@markdown - **ipTM** (interface predicted TM-score): 結合界面の予測構造信頼度（高いほど良い、0–1）
#@markdown - **TaB_RMSD** (Target-aligned Binder RMSD): ターゲットで重ね合わせた際のバインダーのCα RMSD（低いほど良い、Å）
#@markdown - **pLDDT** (predicted Local Distance Difference Test): バインダーチェインの予測構造信頼度（高いほど良い、0–1）
#@markdown ### AF2 閾値
af2_ipSAE_cutoff = 0.5 #@param {type:"number"}
af2_ipTM_cutoff = 0.7 #@param {type:"number"}
af2_TaB_RMSD_cutoff = 3.0 #@param {type:"number"}
af2_pLDDT_cutoff = 0.8 #@param {type:"number"}
#@markdown ### Boltz-1 閾値
boltz1_ipSAE_cutoff = 0.5 #@param {type:"number"}
boltz1_ipTM_cutoff = 0.7 #@param {type:"number"}
boltz1_TaB_RMSD_cutoff = 3.0 #@param {type:"number"}
boltz1_pLDDT_cutoff = 0.8 #@param {type:"number"}
#@markdown ### Boltz-2 閾値
boltz2_ipSAE_cutoff = 0.5 #@param {type:"number"}
boltz2_ipTM_cutoff = 0.7 #@param {type:"number"}
boltz2_TaB_RMSD_cutoff = 3.0 #@param {type:"number"}
boltz2_pLDDT_cutoff = 0.8 #@param {type:"number"}

import pandas as pd
import numpy as np
from IPython.display import display

csv_path = PROJECT_DIR / "results" / "validation_scores.csv"

if not csv_path.exists():
    print("validation_scores.csv が見つかりません。先に 4.1 を実行してください。")
else:
    df = pd.read_csv(csv_path)

    # Thresholds keyed by prefix: (ipSAE, ipTM, TaB_RMSD, pLDDT)
    cutoffs = {
        "af2": (af2_ipSAE_cutoff, af2_ipTM_cutoff, af2_TaB_RMSD_cutoff, af2_pLDDT_cutoff),
        "boltz1": (boltz1_ipSAE_cutoff, boltz1_ipTM_cutoff, boltz1_TaB_RMSD_cutoff, boltz1_pLDDT_cutoff),
        "boltz2": (boltz2_ipSAE_cutoff, boltz2_ipTM_cutoff, boltz2_TaB_RMSD_cutoff, boltz2_pLDDT_cutoff),
    }
    methods = [(p, l) for p, l in [("af2", "AF2"), ("boltz1", "Boltz-1"), ("boltz2", "Boltz-2")]
               if f"{p}_ipTM" in df.columns]

    # ── Pass / Fail 判定（ipSAE があれば ipSAE、無ければ ipTM を使用）──
    show_cols = ["name", "length"]
    for pfx, _ in methods:
        ipsae_c, iptm_c, rmsd_c, plddt_c = cutoffs[pfx]
        has_ipsae = f"{pfx}_ipSAE" in df.columns
        cond = (df[f"{pfx}_TaB_RMSD"] < rmsd_c) & (df[f"{pfx}_pLDDT_binder"] > plddt_c)
        cond = cond & (df[f"{pfx}_ipSAE"] > ipsae_c if has_ipsae else df[f"{pfx}_ipTM"] > iptm_c)
        df[f"{pfx}_pass"] = cond
        show_cols += [f"{pfx}_pLDDT_binder", f"{pfx}_ipSAE" if has_ipsae else f"{pfx}_ipTM",
                      f"{pfx}_TaB_RMSD", f"{pfx}_pass"]
    show_cols = [c for c in show_cols if c in df.columns]

    def color_pass(val):
        if val == True:
            return "background-color: #d4edda"
        if val == False:
            return "background-color: #f8d7da"
        return ""

    pass_cols = [c for c in show_cols if c.endswith("_pass")]
    styled = df[show_cols].style.map(color_pass, subset=pass_cols).format(precision=3, na_rep="N/A")
    display(styled)

    # ── Summary ──
    print()
    for pfx, label in methods:
        ipsae_c, iptm_c, rmsd_c, plddt_c = cutoffs[pfx]
        n = df[f"{pfx}_pass"].sum()
        crit = f"ipSAE > {ipsae_c}" if f"{pfx}_ipSAE" in df.columns else f"ipTM > {iptm_c}"
        print(f"{label:<8}: {n}/{len(df)} passed  ({crit}, TaB_RMSD < {rmsd_c} Å, pLDDT > {plddt_c})")

    # ── Passed candidates ──
    pass_any = pd.Series(False, index=df.index)
    for pfx, _ in methods:
        pass_any |= df[f"{pfx}_pass"]

    if pass_any.any():
        print("\nPassed candidates:")
        for _, row in df[pass_any].iterrows():
            tags = [label for pfx, label in methods if row.get(f"{pfx}_pass")]
            print(f"  {row['name']}  ({', '.join(tags)})")
    else:
        print("\nPass した候補がありません。閾値を緩和して再実行してください。")


In [ ]:
#@title 4.4 リフォールディング結果の3D表示
#@markdown フィルタリング結果の予測構造を一覧表示します。
#@markdown Pass した候補を先頭に、それ以降に残りの候補を ipSAE 降順で並べます。
#@markdown 設計構造のバインダーを半透明で重ね合わせ表示し、予測構造とのずれを確認できます。
#@markdown
#@markdown ---
#@markdown - **view_method**: 表示するモデルの予測構造
#@markdown - **max_cols**: グリッドの最大列数（1–8）
#@markdown - **linked_view**: ON にすると全ビューアが連動して回転・ズームします
#@markdown ---
view_method = "AF2" #@param ["AF2", "Boltz-1", "Boltz-2"]
max_cols = 4 #@param {type:"integer"}
linked_view = True #@param {type:"boolean"}

import math
import pandas as pd
import numpy as np
import py3Dmol
from pathlib import Path

csv_path = PROJECT_DIR / "results" / "validation_scores.csv"
SAMPLES_DIR = PROJECT_DIR / "results" / "samples"

if not csv_path.exists():
    print("validation_scores.csv が見つかりません。先に 4.1 を実行してください。")
else:
    df = pd.read_csv(csv_path)
    pfx = {"AF2": "af2", "Boltz-1": "boltz1", "Boltz-2": "boltz2"}[view_method]
    pass_col = f"{pfx}_pass"
    ipsae_col = f"{pfx}_ipSAE"
    iptm_col = f"{pfx}_ipTM"

    if ipsae_col in df.columns:
        sort_col = ipsae_col
    elif iptm_col in df.columns:
        sort_col = iptm_col
    else:
        sort_col = None

    has_pass = pass_col in df.columns
    if has_pass and sort_col:
        df_sorted = df.sort_values([pass_col, sort_col], ascending=[False, False], na_position="last")
    elif has_pass:
        df_sorted = df.sort_values(pass_col, ascending=False, na_position="last")
    elif sort_col:
        df_sorted = df.sort_values(sort_col, ascending=False, na_position="last")
    else:
        df_sorted = df

    design_pdbs = {p.stem: p for p in sorted(SAMPLES_DIR.rglob("*.pdb"))}

    def align_pred_to_design(des_pdb_path, pred_pdb_path):
        """Align predicted structure onto design coordinates using target CA atoms."""
        des_struct, target_cids, des_bcid = parse_complex(des_pdb_path)
        pred_struct, _, pred_bcid = parse_complex(pred_pdb_path)
        superpose_on_target(des_struct, pred_struct, target_cids)
        return struct_to_pdb(pred_struct), des_bcid, pred_bcid, target_cids

    # Resolve predicted PDB paths
    pred_entries = []
    for _, row in df_sorted.iterrows():
        name = row["name"]
        if view_method == "AF2":
            pred_pdb = PROJECT_DIR / "results" / "af2" / f"{name}_af2.pdb"
        else:
            pred_dir = PROJECT_DIR / "results" / pfx / "boltz_results_input" / "predictions" / name
            pred_pdbs_list = sorted(pred_dir.glob("*.pdb")) if pred_dir.exists() else []
            pred_pdb = pred_pdbs_list[0] if pred_pdbs_list else None

        if pred_pdb and pred_pdb.exists():
            score = row.get(sort_col, float("nan")) if sort_col else float("nan")
            passed = bool(row.get(pass_col, False)) if has_pass else None
            pred_entries.append({"name": name, "path": pred_pdb, "score": score, "passed": passed})

    if not pred_entries:
        print(f"{view_method} の予測構造が見つかりません。先に 4.1 を実行してください。")
    else:
        n = len(pred_entries)
        cols = min(max_cols, n, 8)
        rows = min(math.ceil(n / cols), 8)
        n_show = rows * cols

        view = py3Dmol.view(viewergrid=(rows, cols), width=250 * cols, height=280 * rows, linked=linked_view)

        score_label = "ipSAE" if sort_col and "ipSAE" in sort_col else "ipTM"
        for idx, entry in enumerate(pred_entries[:n_show]):
            r, c = divmod(idx, cols)
            des_pdb_path = design_pdbs.get(entry["name"])

            if des_pdb_path:
                # Model 0: design structure
                with open(des_pdb_path) as f:
                    des_data = f.read()
                view.addModel(des_data, "pdb", viewer=(r, c))

                # Model 1: predicted structure (aligned to design)
                aligned_pred, des_binder_cid, pred_binder_cid, des_target_cids = align_pred_to_design(
                    des_pdb_path, entry["path"])
                view.addModel(aligned_pred, "pdb", viewer=(r, c))

                # Design: target white, hotspot red, binder transparent gray
                for cid in des_target_cids:
                    view.setStyle({"model": 0, "chain": cid},
                                  {"cartoon": {"color": "white"}}, viewer=(r, c))
                if hotspot_resis:
                    view.setStyle(
                        {"model": 0, "chain": target_chain, "resi": hotspot_resis},
                        {"cartoon": {"color": "red"}, "stick": {"color": "red"}},
                        viewer=(r, c),
                    )
                view.setStyle({"model": 0, "chain": des_binder_cid},
                              {"cartoon": {"color": "lightgray", "opacity": 0.5}}, viewer=(r, c))

                # Predicted: hide target, show binder
                view.setStyle({"model": 1}, {}, viewer=(r, c))
                view.setStyle({"model": 1, "chain": pred_binder_cid},
                              {"cartoon": {"colorscheme": {"prop": "b",
                                           "gradient": "roygb", "min": 0, "max": 100}}},
                              viewer=(r, c))
            else:
                # Fallback: predicted structure only
                with open(entry["path"]) as f:
                    pdb_data = f.read()
                view.addModel(pdb_data, "pdb", viewer=(r, c))
                _, pred_target_cids, pred_binder_cid = parse_complex(entry["path"])
                for cid in pred_target_cids:
                    view.setStyle({"model": 0, "chain": cid},
                                  {"cartoon": {"color": "white"}}, viewer=(r, c))
                view.setStyle({"model": 0, "chain": pred_binder_cid},
                              {"cartoon": {"colorscheme": {"prop": "b",
                                           "gradient": "roygb", "min": 0, "max": 100}}},
                              viewer=(r, c))

            score_str = f"{entry['score']:.3f}" if not np.isnan(entry["score"]) else "N/A"
            tag = "PASS" if entry["passed"] else "FAIL"
            label_text = f"{entry['name']}\n{score_label}={score_str} [{tag}]" if has_pass else f"{entry['name']}\n{score_label}={score_str}"
            view.addLabel(label_text, {
                "position": {"x": 0, "y": 0, "z": 0},
                "backgroundColor": "black", "fontColor": "white",
                "fontSize": 10, "backgroundOpacity": 0.6,
                "inFront": True,
            }, viewer=(r, c))
            view.zoomTo(viewer=(r, c))

        view.show()

        mode = "linked" if linked_view else "independent"
        n_passed = sum(1 for e in pred_entries if e["passed"]) if has_pass else n
        print(f"{view_method}: {min(n, n_show)}/{n} designs ({n_passed} passed), sorted by {score_label} desc  ({mode})")
        print(f"White: target, Red: hotspot, Gray(transparent): design binder, Orange→Blue: predicted binder (confidence)")
        if n > n_show:
            print(f"  残り {n - n_show} 件は表示されていません。")

In [ ]:
#@title 5. 結果のダウンロード
#@markdown プロジェクトの全結果（デザインPDB、予測構造、スコアCSV）をzipにまとめてダウンロードします。
#@markdown
#@markdown ---
import shutil
from google.colab import files

results_dir = PROJECT_DIR / "results"
if not results_dir.exists() or not any(results_dir.iterdir()):
    print("結果ファイルが見つかりません。先にパイプラインを実行してください。")
else:
    zip_name = f"{project_name}_results"
    zip_path = PROJECT_DIR / zip_name
    shutil.make_archive(str(zip_path), "zip", root_dir=str(PROJECT_DIR), base_dir="results")
    zip_file = f"{zip_path}.zip"

    # Summary
    pdb_count = len(list(results_dir.rglob("*.pdb")))
    csv_count = len(list(results_dir.rglob("*.csv")))
    size_mb = Path(zip_file).stat().st_size / (1024 * 1024)
    print(f"Archive: {Path(zip_file).name} ({size_mb:.1f} MB)")
    print(f"  PDB files: {pdb_count}")
    print(f"  CSV files: {csv_count}")
    print()

    files.download(zip_file)
